# 00 Data Check

元データの構造を確認し、Step 0 の前処理出力を作るNotebook。

## 目的

- 設定セルで `MATCH_DATE`、`MATCH_ID`、`MATCH_LABEL` を指定する
- `DATASET_NAME = 日付_試合ID_試合名` の形でフォルダ名を統一する
- トラッキングXMLから各フレームのチーム重心を作る
- メタデータXMLからチーム、選手、GKを確認する
- GKを除外する
- 座標を 0〜1 の端基準から -0.5〜0.5 の中央基準に変換する
- 各チーム10人の平均座標を重心として計算する
- `ballStatus` が `HOME` / `AWAY` のフレームを攻撃/守備ラベルに変換する
- `data/processed/{DATASET_NAME}/team_centroids_by_frame.csv` を保存する
- `data/processed/{DATASET_NAME}/team_centroids_by_sequence.csv` を保存する

## 設定

新しい試合を分析する場合は、このセルの `MATCH_DATE`、`MATCH_ID`、`MATCH_LABEL` だけ変更する。

`COORD_MODE = 'centered'` の場合、トラッキングデータの座標から `0.5` を引いて、フィールド中央を原点にしてから重心を計算する。

重み付き重心を使う場合は、`CENTROID_METHOD = 'weighted'` にして `POSITION_WEIGHTS` を変更する。

特定の時間帯だけ分析したい場合は、`START_TIME_SECONDS` と `END_TIME_SECONDS` に秒数を入れる。全時間を使う場合は `None` のままにする。

In [ ]:
import ast
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

def find_project_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists() and (candidate / 'はるた').exists():
            return candidate
    raise FileNotFoundError('プロジェクトルートが見つかりません。data/ と はるた/ がある場所で実行してください。')


PROJECT_ROOT = find_project_root()
HARUTA_ROOT = PROJECT_ROOT / 'はるた'

# ここを変更すれば、分析対象や分析条件を切り替えられる。
MATCH_DATE = '2023-11-25'
MATCH_ID = '117093'
MATCH_LABEL = 'tsukuba_vs_tsukuba-b'
# 例: 20分〜23分だけ分析する場合は 20 * 60, 23 * 60 にする。
# 全時間を使う場合は None のままにする。
START_TIME_SECONDS = None
END_TIME_SECONDS = None

# 座標は 0〜1 の端基準から、-0.5〜0.5 の中央基準に変換してから重心を計算する。
COORD_MODE = 'centered'
X_HALF_LINE = 0.5
Y_HALF_LINE = 0.5
# simple_mean: 全選手を同じ重みで平均する
# weighted: POSITION_WEIGHTS の重みで平均する
CENTROID_METHOD = 'simple_mean'
# metadata XML の position 表記に合わせて設定する。
# すべて 1.0 なら simple_mean と同じ。仮説に合わせて値を変更する。
POSITION_WEIGHTS = {
    'CB': 1.0,
    'LB': 1.0,
    'RB': 1.0,
    'CM': 1.0,
    'CAM': 1.0,
    'LM': 1.0,
    'RM': 1.0,
    'CF': 1.0,
    'LW': 1.0,
    'RW': 1.0,
}
UNKNOWN_POSITION_WEIGHT = 1.0
VALID_POSSESSION_STATUSES = ('HOME', 'AWAY')

DATASET_NAME = f'{MATCH_DATE}_{MATCH_ID}_{MATCH_LABEL}'
DATASET_DIR = PROJECT_ROOT / 'data' / 'candidates' / '01_sorted' / DATASET_NAME
TRACKING_XML = DATASET_DIR / '01_tracking_core' / f'{MATCH_ID}_tracker_box_data.xml'
METADATA_XML = DATASET_DIR / '03_player_master' / f'{MATCH_ID}_tracker_box_metadata.xml'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / DATASET_NAME

DATASET_NAME, START_TIME_SECONDS, END_TIME_SECONDS, COORD_MODE, TRACKING_XML, METADATA_XML, PROCESSED_DIR


## 関数定義

このNotebook内で使う処理をここにまとめる。

In [ ]:
def load_tracking_metadata(metadata_xml_path):
    root = ET.parse(metadata_xml_path).getroot()
    pitch = root.find('pitch')
    pitch_info = {'width': float(pitch.attrib['width']), 'height': float(pitch.attrib['height'])}

    teams = []
    for team in root.findall('.//teams/team'):
        teams.append({
            'team_id': str(team.attrib['id']),
            'team_name': team.attrib['name'],
            'team_name_en': team.attrib.get('nameEn', ''),
            'side': team.attrib['side'],
            'team_label': 'A' if team.attrib['side'] == 'away' else 'B',
        })

    players = []
    for player in root.findall('.//players/player'):
        players.append({
            'player_id': str(player.attrib['id']),
            'player_name': player.attrib.get('name', ''),
            'shirt_number': player.attrib.get('shirtNumber', ''),
            'position': player.attrib.get('position', ''),
            'team_id': str(player.attrib['teamId']),
            'team_name': player.attrib.get('teamName', ''),
        })

    teams_df = pd.DataFrame(teams)
    players_df = pd.DataFrame(players).merge(
        teams_df[['team_id', 'side', 'team_label']],
        on='team_id',
        how='left',
    )
    return {'pitch': pitch_info, 'teams': teams_df, 'players': players_df}


def parse_loc(loc_text):
    x, y = ast.literal_eval(loc_text)
    return float(x), float(y)


def convert_coordinates(x, y):
    if COORD_MODE == 'raw':
        return x, y
    if COORD_MODE == 'centered':
        return x - X_HALF_LINE, y - Y_HALF_LINE
    raise ValueError(f'未対応のCOORD_MODEです: {COORD_MODE}')


def get_player_weight(player_info):
    if CENTROID_METHOD == 'simple_mean':
        return 1.0
    if CENTROID_METHOD == 'weighted':
        return float(POSITION_WEIGHTS.get(player_info.get('position'), UNKNOWN_POSITION_WEIGHT))
    raise ValueError(f'未対応のCENTROID_METHODです: {CENTROID_METHOD}')


def is_in_time_window(match_time_ms):
    match_time_seconds = match_time_ms / 1000
    if START_TIME_SECONDS is not None and match_time_seconds < START_TIME_SECONDS:
        return False
    if END_TIME_SECONDS is not None and match_time_seconds > END_TIME_SECONDS:
        return False
    return True


def build_centroid_frames(tracking_xml_path, metadata_xml_path, match_id):
    metadata = load_tracking_metadata(metadata_xml_path)
    players_by_id = metadata['players'].set_index('player_id').to_dict('index')
    teams_by_id = metadata['teams'].set_index('team_id').to_dict('index')

    rows = []
    current_sequence_id = 0
    previous_sequence_key = None

    for _, elem in ET.iterparse(tracking_xml_path, events=('end',)):
        if elem.tag != 'frame':
            continue

        ball_status = elem.attrib.get('ballStatus')
        if ball_status not in VALID_POSSESSION_STATUSES:
            elem.clear()
            continue

        event_period = elem.attrib.get('eventPeriod')
        match_time = int(float(elem.attrib['matchTime']))
        frame_number = int(float(elem.attrib['frameNumber']))
        if not is_in_time_window(match_time):
            elem.clear()
            continue

        possession_side = ball_status.lower()

        sequence_key = (event_period, ball_status)
        if sequence_key != previous_sequence_key:
            current_sequence_id += 1
            previous_sequence_key = sequence_key

        sums = {}
        for player in elem.findall('player'):
            player_id = str(player.attrib['playerId'])
            player_info = players_by_id.get(player_id)
            if not player_info or player_info.get('position') == 'GK':
                continue

            team_id = str(player_info['team_id'])
            raw_x, raw_y = parse_loc(player.attrib['loc'])
            x, y = convert_coordinates(raw_x, raw_y)
            weight = get_player_weight(player_info)
            sums.setdefault(team_id, {'x': 0.0, 'y': 0.0, 'n': 0, 'weight_sum': 0.0})
            sums[team_id]['x'] += x * weight
            sums[team_id]['y'] += y * weight
            sums[team_id]['n'] += 1
            sums[team_id]['weight_sum'] += weight

        for team_id, values in sums.items():
            if values['n'] != 10:
                continue

            team_info = teams_by_id[team_id]
            side = team_info['side']
            phase = 'attack' if side == possession_side else 'defense'
            team_label = team_info['team_label']
            rows.append({
                'match_id': match_id,
                'sequence_id': current_sequence_id,
                'match_time': match_time,
                'frame_number': frame_number,
                'event_period': event_period,
                'ball_status': ball_status,
                'possession_side': possession_side,
                'team_id': team_id,
                'team_name': team_info['team_name'],
                'team_label': team_label,
                'phase': phase,
                'group': f'{team_label}_{phase}',
                'coord_mode': COORD_MODE,
                'centroid_method': CENTROID_METHOD,
                'start_time_seconds': START_TIME_SECONDS,
                'end_time_seconds': END_TIME_SECONDS,
                'centroid_x': values['x'] / values['weight_sum'],
                'centroid_y': values['y'] / values['weight_sum'],
                'n_players': values['n'],
                'weight_sum': values['weight_sum'],
            })

        elem.clear()

    return pd.DataFrame(rows)


def aggregate_centroid_sequences(team_centroids_by_frame):
    group_cols = [
        'match_id', 'sequence_id', 'event_period', 'ball_status', 'possession_side',
        'team_id', 'team_name', 'team_label', 'phase', 'group',
        'coord_mode', 'centroid_method', 'start_time_seconds', 'end_time_seconds',
    ]
    team_centroids_by_sequence = (
        team_centroids_by_frame.groupby(group_cols, as_index=False, dropna=False)
        .agg(
            mean_centroid_x=('centroid_x', 'mean'),
            mean_centroid_y=('centroid_y', 'mean'),
            n_frames=('frame_number', 'count'),
            mean_weight_sum=('weight_sum', 'mean'),
            start_match_time=('match_time', 'min'),
            end_match_time=('match_time', 'max'),
        )
        .sort_values(['sequence_id', 'team_label'])
        .reset_index(drop=True)
    )
    team_centroids_by_sequence['duration_seconds'] = (
        team_centroids_by_sequence['end_match_time']
        - team_centroids_by_sequence['start_match_time']
    ) / 1000
    return team_centroids_by_sequence


## メタデータ確認

In [ ]:
metadata = load_tracking_metadata(METADATA_XML)
metadata['teams']

In [ ]:
players = metadata['players']
players.groupby(['team_label', 'team_id', 'team_name', 'position']).size().reset_index(name='players').head(20)

## Step 0 出力作成

`ballStatus` の `HOME` / `AWAY` をボール保持側として扱う。保持側のチームを `attack`、もう一方を `defense` とする。`BALLOUT` / `NEUTRAL` は検定対象から除外する。

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

team_centroids_by_frame = build_centroid_frames(TRACKING_XML, METADATA_XML, MATCH_ID)
team_centroids_by_sequence = aggregate_centroid_sequences(team_centroids_by_frame)

frame_output_path = PROCESSED_DIR / 'team_centroids_by_frame.csv'
sequence_output_path = PROCESSED_DIR / 'team_centroids_by_sequence.csv'

team_centroids_by_frame.to_csv(frame_output_path, index=False)
team_centroids_by_sequence.to_csv(sequence_output_path, index=False)

print(f'Saved frame data: {frame_output_path}')
print(f'Saved sequence data: {sequence_output_path}')
print(f'frame rows: {len(team_centroids_by_frame):,}')
print(f'sequence rows: {len(team_centroids_by_sequence):,}')

## 出力データ確認

In [ ]:
team_centroids_by_frame.head()

In [ ]:
team_centroids_by_frame.groupby(['group', 'n_players']).size().reset_index(name='frames')

In [ ]:
team_centroids_by_sequence.head()

In [ ]:
team_centroids_by_sequence.groupby('group').agg(
    sequences=('sequence_id', 'count'),
    mean_x=('mean_centroid_x', 'mean'),
    mean_y=('mean_centroid_y', 'mean'),
    mean_duration_seconds=('duration_seconds', 'mean'),
)